In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

In [2]:
traders = pd.DataFrame({
    "trader_id": range(1, 16),
    "trader_name": [
        "Alex Morgan",
        "Maria Rossi",
        "John Smith",
        "Elena Papadopoulos",
        "Thomas Weber",
        "Sofia Garcia",
        "Daniel Brown",
        "Anna Muller",
        "Nikos Georgiou",
        "Laura Costa",
        "Mark Taylor",
        "Giulia Romano",
        "Peter Novak",
        "Emma Wilson",
        "George Adams"
    ],
    "desk": [
        "Power Trading",
        "Renewables",
        "Risk Management",
        "Power Trading",
        "Gas Trading",
        "Renewables",
        "Quant Trading",
        "Power Trading",
        "Risk Management",
        "Renewables",
        "Gas Trading",
        "Power Trading",
        "Quant Trading",
        "Risk Management",
        "Power Trading"
    ],
    "country": [
        "Greece",
        "Italy",
        "Germany",
        "Greece",
        "Germany",
        "Spain",
        "UK",
        "Germany",
        "Greece",
        "Italy",
        "UK",
        "Italy",
        "Austria",
        "France",
        "Greece"
    ]
})

traders.head()

,trader_id,trader_name,desk,country
0,1,Alex Morgan,Power Trading,Greece
1,2,Maria Rossi,Renewables,Italy
2,3,John Smith,Risk Management,Germany
3,4,Elena Papadopoulos,Power Trading,Greece
4,5,Thomas Weber,Gas Trading,Germany


In [3]:
traders.to_csv("traders.csv", index=False)

In [4]:
markets = pd.DataFrame({
    "market_id": [1,2,3,4,5],
    "market_name": [
        "Day Ahead Market",
        "Intraday Market",
        "Balancing Market",
        "Forward Market",
        "Ancillary Services"
    ],
    "market_type": [
        "DAM",
        "IDM",
        "BM",
        "Forward",
        "Reserve"
    ],
    "country": [
        "Greece",
        "Greece",
        "Greece",
        "Italy",
        "Germany"
    ]
})

markets

,market_id,market_name,market_type,country
0,1,Day Ahead Market,DAM,Greece
1,2,Intraday Market,IDM,Greece
2,3,Balancing Market,BM,Greece
3,4,Forward Market,Forward,Italy
4,5,Ancillary Services,Reserve,Germany


In [5]:
markets.to_csv("markets.csv", index=False)

In [6]:
assets = pd.DataFrame({
    "asset_id": range(1,11),
    "asset_name": [
        "Wind Farm Alpha",
        "Wind Farm Beta",
        "Solar Park Gamma",
        "Solar Park Delta",
        "Gas Plant Omega",
        "Gas Plant Sigma",
        "Hydro Plant A",
        "Hydro Plant B",
        "Battery Storage X",
        "Battery Storage Y"
    ],
    "asset_type": [
        "Wind",
        "Wind",
        "Solar",
        "Solar",
        "Gas",
        "Gas",
        "Hydro",
        "Hydro",
        "Storage",
        "Storage"
    ],
    "capacity_mw": [
        120,
        80,
        60,
        100,
        300,
        250,
        150,
        90,
        50,
        70
    ],
    "renewable": [
        1,1,1,1,0,0,1,1,0,0
    ]
})

assets

,asset_id,asset_name,asset_type,capacity_mw,renewable
0,1,Wind Farm Alpha,Wind,120,1
1,2,Wind Farm Beta,Wind,80,1
2,3,Solar Park Gamma,Solar,60,1
3,4,Solar Park Delta,Solar,100,1
4,5,Gas Plant Omega,Gas,300,0
5,6,Gas Plant Sigma,Gas,250,0
6,7,Hydro Plant A,Hydro,150,1
7,8,Hydro Plant B,Hydro,90,1
8,9,Battery Storage X,Storage,50,0
9,10,Battery Storage Y,Storage,70,0


In [7]:
assets.to_csv("assets.csv", index=False)

In [8]:
## Generate hourly electricity prices

import pandas as pd
import numpy as np
from datetime import datetime

np.random.seed(42)

dates = pd.date_range(
    start="2025-01-01",
    end="2025-12-31 23:00",
    freq="h"
)

prices = []

for dt in dates:

    hour = dt.hour
    month = dt.month

    # Base price
    price = 70


    # Night hours
    if 0 <= hour < 6:
        price -= 15


    # Morning peak
    if 8 <= hour <= 11:
        price += 30


    # Evening peak
    if 18 <= hour <= 22:
        price += 50


    # Winter demand
    if month in [1,2,12]:
        price += 25


    # Summer demand
    if month in [7,8]:
        price += 20


    # Random market variation
    price += np.random.normal(0,10)


    # Rare price spikes
    if np.random.random() < 0.02:
        price += np.random.randint(50,150)


    # Rare negative prices
    if np.random.random() < 0.005:
        price -= np.random.randint(50,100)


    prices.append(round(price,2))


market_prices_df = pd.DataFrame({

    "market_id":1,
    "price_timestamp":dates,
    "settlement_price":prices

})


market_prices_df.head()

,market_id,price_timestamp,settlement_price
0,1,2025-01-01 00:00:00,84.97
1,1,2025-01-01 01:00:00,78.62
2,1,2025-01-01 02:00:00,95.79
3,1,2025-01-01 03:00:00,87.67
4,1,2025-01-01 04:00:00,75.37


In [12]:
market_prices_df["hour"] = market_prices_df["price_timestamp"].dt.hour

In [13]:
def classify_peak(hour):
    if 8 <= hour <= 11 or 18 <= hour <= 22:
        return "Peak"
    else:
        return "Off-Peak"


market_prices_df["peak_type"] = (
    market_prices_df["hour"]
    .apply(classify_peak)
)

In [14]:
market_prices_df.head()

,market_id,price_timestamp,settlement_price,hour,peak_type
0,1,2025-01-01 00:00:00,84.97,0,Off-Peak
1,1,2025-01-01 01:00:00,78.62,1,Off-Peak
2,1,2025-01-01 02:00:00,95.79,2,Off-Peak
3,1,2025-01-01 03:00:00,87.67,3,Off-Peak
4,1,2025-01-01 04:00:00,75.37,4,Off-Peak


In [15]:
market_prices_df.to_csv(
    "market_prices.csv",
    index=False
)

In [16]:
## Generate simulated energy trades
market_prices = pd.read_csv(
    "market_prices.csv",
    parse_dates=["price_timestamp"]
)

market_prices.head()

,market_id,price_timestamp,settlement_price,hour,peak_type
0,1,2025-01-01 00:00:00,84.97,0,Off-Peak
1,1,2025-01-01 01:00:00,78.62,1,Off-Peak
2,1,2025-01-01 02:00:00,95.79,2,Off-Peak
3,1,2025-01-01 03:00:00,87.67,3,Off-Peak
4,1,2025-01-01 04:00:00,75.37,4,Off-Peak


In [17]:
#trades creation
np.random.seed(42)

n_trades = 50000


trade_dates = np.random.choice(
    market_prices["price_timestamp"],
    size=n_trades
)


trades = pd.DataFrame({

    "trader_id": np.random.randint(
        1,16,
        size=n_trades
    ),

    "market_id": np.random.randint(
        1,6,
        size=n_trades
    ),

    "asset_id": np.random.randint(
        1,11,
        size=n_trades
    ),

    "trade_timestamp": trade_dates

})

In [18]:
trades["trade_type"] = np.random.choice(
    ["BUY","SELL"],
    size=n_trades,
    p=[0.55,0.45]
)

In [20]:

trades["volume_mwh"] = np.random.randint(
    10,
    500,
    size=n_trades
)

In [21]:
trades = trades.merge(
    market_prices[
        [
            "price_timestamp",
            "settlement_price"
        ]
    ],
    left_on="trade_timestamp",
    right_on="price_timestamp",
    how="left"
)

In [22]:
trades["price"] = (
    trades["settlement_price"]
    *
    np.random.uniform(
        0.95,
        1.05,
        size=n_trades
    )
).round(2)

In [23]:
trades = trades[
    [
        "trader_id",
        "market_id",
        "asset_id",
        "trade_timestamp",
        "trade_type",
        "volume_mwh",
        "price"
    ]
]

In [24]:
trades["trade_date"] = (
    trades["trade_timestamp"]
    .dt.date
)

In [25]:
trades = trades[
[
"trader_id",
"market_id",
"asset_id",
"trade_date",
"volume_mwh",
"price",
"trade_type",
"trade_timestamp"
]
]

In [26]:
trades.to_csv(
    "trades.csv",
    index=False
)

In [27]:
trades.shape

(50000, 8)